# 模块十一：主题域 DWD 重组 — Background §6.11 升级

## 学习目标

1. 理解从「按源系统分 DWD」（`dwd/sap_erp/dwd_vbak`）升级为「按业务主题分 DWD」（`dwd/sales/dwd_vbak`）的动机与收益。
2. 跑过 `scripts/restructure_dwd.py` 把 6 张 DWD 表 dual-write 到 4 个主题目录（sales / production / coal_quality / finance），旧 system-分区保留。
3. 验证 dual-write 落盘数据完全一致（旧 `dwd/sap_erp/dwd_vbak` 与新 `dwd/sales/dwd_vbak` 行数严格相等）。
4. 看 DataHub UI 的 `dwd` 自定义 platform 树形浏览（playwright 截图）。

## 与 module1 / module4 的关系

| 维度 | module1 / module4 (Phase 1) | module11 (Phase 2 / 6.11) |
|---|---|---|
| DWD 组织 | `dwd/sap_erp/dwd_vbak`（按源系统分） | `dwd/sales/dwd_vbak`（按业务主题分） |
| 维度表 | 同上 `_dimensions/` 不动 | 同上 `_dimensions/` 不动（共享维度全局可用） |
| DWA 上游 | 读 system-分区 | **仍读 system-分区**（6.12 才切） |
| DataHub 平台 | `sap_erp` / `pi_system` / `lims` / `oa` | 新增 `dwd` 自定义 platform，6 张新表按 `dwd.sales.*` 等命名 |
| 存储成本 | 1× | 2×（dual-write 阶段）；6.12 切流后回 1× |

**核心不变**：`dwd_vbak` 等表名保持不变，仅目录变更。`dwa_*.py` 等下游脚本无需改。

In [ ]:
from pathlib import Path

DWD = Path('..') / 'data' / 'lakehouse' / 'dwd'
print('=== cell 1: 主题域 DWD 目录结构对比 ===\n')

print('▼ 新主题目录 (subject-分区) — 4 个主题 6 张表')
print('-' * 60)
for subject_dir in sorted(p for p in DWD.iterdir() if p.is_dir() and not p.name.startswith('_') and p.name not in {'sap_erp','pi_system','lims','oa'}):
    tables = sorted(t.name for t in subject_dir.iterdir() if t.is_dir())
    print(f'  dwd/{subject_dir.name}/')
    for t in tables:
        print(f'    └── {t}')

print('\n▼ 旧 system-分区 (保留) — 4 个系统 6 张表')
print('-' * 60)
for system_dir in sorted(p for p in DWD.iterdir() if p.is_dir() and p.name in {'sap_erp','pi_system','lims','oa'}):
    tables = sorted(t.name for t in system_dir.iterdir() if t.is_dir())
    print(f'  dwd/{system_dir.name}/')
    for t in tables:
        print(f'    └── {t}')

print('\n▼ 共享维度 (不动) — _dimensions/ 全局共享')
print('-' * 60)
dim_dir = DWD / '_dimensions'
if dim_dir.is_dir():
    for t in sorted(p.name for p in dim_dir.iterdir() if p.is_dir()):
        print(f'  dwd/_dimensions/{t}')

print()
print('=== 表-主题映射 (canonical) ===')
print(f'  {"旧 system-分区":<32} → {"新 subject-分区":<32}')
print('  ' + '-' * 66)
SUBJECT_MAP = [
    ('dwd/sap_erp/dwd_vbak',     'dwd/sales/dwd_vbak'),
    ('dwd/sap_erp/dwd_vbap',     'dwd/sales/dwd_vbap'),
    ('dwd/sap_erp/dwd_kna1',     'dwd/sales/dwd_kna1'),
    ('dwd/pi_system/dwd_tags',   'dwd/production/dwd_tags'),
    ('dwd/lims/dwd_samples',     'dwd/coal_quality/dwd_samples'),
    ('dwd/oa/dwd_doc_flow',      'dwd/finance/dwd_doc_flow'),
]
for old, new in SUBJECT_MAP:
    print(f'  {old:<32} → {new:<32}')

In [ ]:
from pathlib import Path
import duckdb

print('=== cell 2: DuckDB 验证新 vs 旧 行数严格一致 ===\n')

LAKEHOUSE = (Path('..') / 'data' / 'lakehouse').resolve()
con = duckdb.connect()
SUBJECT_MAP = [
    ('dwd/sap_erp/dwd_vbak',     'dwd/sales/dwd_vbak'),
    ('dwd/sap_erp/dwd_vbap',     'dwd/sales/dwd_vbap'),
    ('dwd/sap_erp/dwd_kna1',     'dwd/sales/dwd_kna1'),
    ('dwd/pi_system/dwd_tags',   'dwd/production/dwd_tags'),
    ('dwd/lims/dwd_samples',     'dwd/coal_quality/dwd_samples'),
    ('dwd/oa/dwd_doc_flow',      'dwd/finance/dwd_doc_flow'),
]

print(f'  {"OLD (system-分区)":<48} {"rows":>12}    {"NEW (subject-分区)":<48} {"rows":>12}    {"eq":>4}')
print('  ' + '-' * 132)
all_equal = True
for old, new in SUBJECT_MAP:
    old_count = con.sql(f"SELECT count(*) FROM delta_scan('{LAKEHOUSE}/{old}')").fetchone()[0]
    new_count = con.sql(f"SELECT count(*) FROM delta_scan('{LAKEHOUSE}/{new}')").fetchone()[0]
    eq = old_count == new_count
    all_equal = all_equal and eq
    mark = '✅' if eq else '❌'
    print(f'  {old:<48} {old_count:>12,}    {new:<48} {new_count:>12,}    {mark:>4}')

print()
print(f'  整体: {"✅ 6/6 行数严格一致 — dual-write 落盘正确" if all_equal else "❌ 有差异，需排查"}')

print('\n=== 主题域内同源跨表 (sales 主题) ===')
print('-' * 60)
sales_vbak = con.sql(f"SELECT count(*) FROM delta_scan('{LAKEHOUSE}/dwd/sales/dwd_vbak')").fetchone()[0]
sales_vbap = con.sql(f"SELECT count(*) FROM delta_scan('{LAKEHOUSE}/dwd/sales/dwd_vbap')").fetchone()[0]
sales_kna1 = con.sql(f"SELECT count(*) FROM delta_scan('{LAKEHOUSE}/dwd/sales/dwd_kna1')").fetchone()[0]
print(f'  dwd/sales/dwd_vbak : {sales_vbak:>12,} 行')
print(f'  dwd/sales/dwd_vbap : {sales_vbap:>12,} 行')
print(f'  dwd/sales/dwd_kna1 : {sales_kna1:>12,} 行')
print(f'  sales 主题合计: {sales_vbak + sales_vbap + sales_kna1:,} 行（3 张表，跨系统 SAP 同主题）')

In [ ]:
import time
from pathlib import Path
from playwright.async_api import async_playwright

print('=== cell 3: DataHub UI 主题域 DWD 截图 (Playwright async) ===\n')

UI = 'http://localhost:29002'
OUT_DIR = Path('step_images')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'module11_subject_dwd.png'

URL = f'{UI}/browse/dataset'
print(f'  打开: {URL}')
print(f'  保存: {OUT_PATH.resolve()}')

p = await async_playwright().start()
browser = await p.chromium.launch(headless=True)
ctx = await browser.new_context(viewport={'width': 1600, 'height': 1000})
page = await ctx.new_page()
await page.goto(URL, wait_until='domcontentloaded', timeout=60000)
await page.wait_for_timeout(8000)
await page.screenshot(path=str(OUT_PATH), full_page=True)
await browser.close()
await p.stop()

size_kb = OUT_PATH.stat().st_size / 1024
print(f'  ✅ 截图保存: {OUT_PATH} ({size_kb:.1f} KB)')

print('\n=== 验证 6 张新主题 DWD 表已入 OpenSearch ===')
import requests
r = requests.post(
    'http://localhost:29200/datasetindex_v2/_search',
    json={
        'size': 20,
        'query': {'bool': {'should': [
            {'term': {'id': 'dwd.sales.dwd_vbak'}},
            {'term': {'id': 'dwd.sales.dwd_vbap'}},
            {'term': {'id': 'dwd.sales.dwd_kna1'}},
            {'term': {'id': 'dwd.production.dwd_tags'}},
            {'term': {'id': 'dwd.coal_quality.dwd_samples'}},
            {'term': {'id': 'dwd.finance.dwd_doc_flow'}},
        ], 'minimum_should_match': 1}},
    },
    timeout=10,
)
hits = r.json().get('hits', {}).get('hits', [])
print(f'  OpenSearch datasetindex_v2 dwd 主题表: {len(hits)} 条')
for h in hits:
    src = h['_source']
    print(f'    {src["urn"]}  (name={src["name"]}, browse={src.get("browsePathV2", "")})')
